# Container: Rawdata → Frame Extraction (Unlabelled) — LOCAL

## Strategy
The `rawdata` container (2025) has **~252,000 videos, 2.7TB** across 5 locations.
There is **no prey annotation CSV** for 2025 yet (like for 2024).

Pipeline:
1. Download video from Azure Blob → temp file
2. cv2 motion detection → extract frames where something is happening
3. Sharpness filter → keep only clear frames
4. Save extracted frames locally (deletes temp video immediately)
5. After YOLO is trained on 2024 labelled data → run on these frames to auto-label

**Run one location at a time** (change `LOCATION_FILTER` per session).

**Location sizes (rawdata 2025):**
- Haarle:    58,719 videos
- Meddo:     62,151 videos
- Randwijk:  59,428 videos
- Twisk:     53,209 videos
- Vosseveld: 18,744 videos

**Runs directly in VS Code — no Colab needed.**
Uses all CPU cores (20 threads) for parallel processing.

In [1]:
# # CELL 1 — Install (run once, then comment out)
# %pip install azure-storage-blob opencv-python-headless pandas -q
# print('Done.')

In [2]:
# CELL 2 — Output directories
import os, shutil, platform

# Frames go here. ~800GB free is more than enough.
OUTPUT_DIR = r'C:\owl-prey-frames\2025-rawdata'
LOCAL_DIR  = r'C:\owl-prey-frames\temp'
MANIFEST   = os.path.join(OUTPUT_DIR, 'manifest_rawdata.csv')

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(LOCAL_DIR, exist_ok=True)

print(f'OS:     {platform.system()} {platform.machine()}')
print(f'Output: {OUTPUT_DIR}')
print(f'Temp:   {LOCAL_DIR}')
_, _, free = shutil.disk_usage('C:\\')
print(f'Free:   {free/1024**3:.0f} GB')

OS:     Windows AMD64
Output: C:\owl-prey-frames\2025-rawdata
Temp:   C:\owl-prey-frames\temp
Free:   36 GB


In [3]:
# CELL 3 — Config & Azure
from azure.storage.blob import BlobServiceClient
from dotenv import load_dotenv
import cv2, numpy as np, pandas as pd, json, csv
from pathlib import Path
from datetime import datetime
import threading, os

load_dotenv()  # reads .env from project root
CONNECTION_STRING = os.getenv('AZURE_CONNECTION_STRING')
assert CONNECTION_STRING, 'Set AZURE_CONNECTION_STRING in your .env file!'

blob_service     = BlobServiceClient.from_connection_string(CONNECTION_STRING)
container_client = blob_service.get_container_client('rawdata')

MOTION_THRESHOLD  = 25
MIN_MOTION_PIXELS = 5000
SKIP_FRAMES       = 4
MIN_SHARPNESS     = 40

print('Connected to rawdata (2025).')

Connected to rawdata (2025).


In [4]:
# CELL 4 — Pick location
LOCATION_FILTER = 'Vosseveld'  # <-- CHANGE THIS per run

CHECKPOINT_FILE = os.path.join(OUTPUT_DIR, f'checkpoint_rawdata_{LOCATION_FILTER}.json')

def load_checkpoint():
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE) as f: return set(json.load(f))
    return set()

def save_checkpoint(done):
    with open(CHECKPOINT_FILE, 'w') as f: json.dump(list(done), f)

done_blobs = load_checkpoint()

print(f'Listing {LOCATION_FILTER} videos...')
all_blobs = [
    b for b in container_client.list_blobs(name_starts_with=LOCATION_FILTER + '/')
    if b.name.lower().endswith('.mp4') and b.size > 0
]
remaining = [b for b in all_blobs if b.name not in done_blobs]

print(f'Location  : {LOCATION_FILTER}')
print(f'Total     : {len(all_blobs):,} videos  ({sum(b.size for b in all_blobs)/1024**3:.1f} GB)')
print(f'Done      : {len(all_blobs)-len(remaining):,}')
print(f'Remaining : {len(remaining):,}  ({sum(b.size for b in remaining)/1024**3:.1f} GB)')

est_min = len(remaining) * 30 / 60
print(f'\nEst. time (single-thread): {est_min/60:.0f}h {est_min%60:.0f}min')
print('Parallel run will be much faster.')

Listing Vosseveld videos...
Location  : Vosseveld
Total     : 18,738 videos  (185.1 GB)
Done      : 0
Remaining : 18,738  (185.1 GB)

Est. time (single-thread): 156h 9min
Parallel run will be much faster.


In [5]:
# CELL 5 — Load worker module
import multiprocessing
# Now pulling from our dedicated processing script rather than the old worker.py
from owl_prey_detection.data_processing.extraction_worker import process_blob_mp

AZURE_KEY = CONNECTION_STRING.split('AccountKey=')[1].split(';')[0]

print(f'Worker module loaded. CPU cores: {multiprocessing.cpu_count()}')


Worker module loaded. CPU cores: 20


In [ ]:
# CELL 7 — RUN (MULTI-PROCESS — real parallelism, all cores, all RAM)
from concurrent.futures import ProcessPoolExecutor, as_completed
from functools import partial

MAX_WORKERS = max(1, multiprocessing.cpu_count() - 2)  # 20 on your machine

# Freeze the config params so we only pass blob_name per task
_worker = partial(process_blob_mp,
                  azure_key=AZURE_KEY,
                  output_dir=OUTPUT_DIR,
                  local_dir=LOCAL_DIR,
                  skip_frames=SKIP_FRAMES,
                  motion_threshold=MOTION_THRESHOLD,
                  min_motion_pixels=MIN_MOTION_PIXELS,
                  min_sharpness=MIN_SHARPNESS)

total_saved  = 0
total_errors = 0
processed_count = 0
start_time   = datetime.now()

print(f'Starting {MAX_WORKERS} PROCESSES at {start_time.strftime("%H:%M:%S")}')
print(f'Processing {len(remaining):,} videos...\n')

with ProcessPoolExecutor(max_workers=MAX_WORKERS) as executor:
    future_to_name = {
        executor.submit(_worker, b.name): b.name
        for b in remaining
    }

    for future in as_completed(future_to_name):
        blob_name = future_to_name[future]
        processed_count += 1

        try:
            blob_name_ret, n, payload = future.result()
        except Exception as e:
            total_errors += 1
            print(f'[{processed_count:05d}/{len(remaining)}] {Path(blob_name).name} -> CRASH: {e}')
            continue

        if n >= 0:
            done_blobs.add(blob_name_ret)
            total_saved += n
            if isinstance(payload, list) and payload:
                write_header = not os.path.exists(MANIFEST)
                with open(MANIFEST, 'a', newline='') as f:
                    w = csv.DictWriter(f, fieldnames=payload[0].keys())
                    if write_header: w.writeheader()
                    w.writerows(payload)
            print(f'[{processed_count:05d}/{len(remaining)}] {Path(blob_name).name} -> {n} frames')
        else:
            total_errors += 1
            print(f'[{processed_count:05d}/{len(remaining)}] {Path(blob_name).name} -> ERROR: {payload}')

        if processed_count % 10 == 0:
            save_checkpoint(done_blobs)
            elapsed = (datetime.now() - start_time).total_seconds() / 60
            rate    = processed_count / elapsed if elapsed > 0 else 0
            eta     = (len(remaining) - processed_count) / rate if rate > 0 else 0
            print(f'  --- checkpoint | {elapsed:.0f}min | ~{rate:.1f} vids/min | ETA {eta/60:.1f}h ---')

save_checkpoint(done_blobs)
elapsed = (datetime.now() - start_time).total_seconds() / 60
print(f'\n{LOCATION_FILTER} completed in {elapsed:.0f}min | Frames: {total_saved:,} | Errors: {total_errors}')

Starting 18 PROCESSES at 19:15:17
Processing 18,738 videos...

[00001/18738] Vosseveld_ch1_main_20250401013303_20250401013340.mp4 -> 3 frames
[00002/18738] Vosseveld_ch1_main_20250401054238_20250401054317.mp4 -> 3 frames
[00003/18738] Vosseveld_ch1_main_20250401054743_20250401054822.mp4 -> 5 frames
[00004/18738] Vosseveld_ch1_main_20250401055150_20250401055229.mp4 -> 2 frames
[00005/18738] Vosseveld_ch1_main_20250401053359_20250401053437.mp4 -> 6 frames
[00006/18738] Vosseveld_ch1_main_20250401012353_20250401012433.mp4 -> 7 frames
[00007/18738] Vosseveld_ch1_main_20250401013105_20250401013148.mp4 -> 10 frames
[00008/18738] Vosseveld_ch1_main_20250401064004_20250401064050.mp4 -> 19 frames
[00009/18738] Vosseveld_ch1_main_20250401054128_20250401054221.mp4 -> 16 frames
[00010/18738] Vosseveld_ch1_main_20250401064247_20250401064329.mp4 -> 4 frames
  --- checkpoint | 2min | ~5.2 vids/min | ETA 60.2h ---
[00011/18738] Vosseveld_ch1_main_20250401013158_20250401013258.mp4 -> 7 frames
[00012/18

In [ ]:
# CELL 7 — Summary across all locations
print('=== RAWDATA FRAMES BY LOCATION ===')
total = 0
for d in sorted(Path(OUTPUT_DIR).iterdir()):
    if not d.is_dir(): continue
    imgs = list(d.glob('*.jpg'))
    mb   = sum(f.stat().st_size for f in imgs) / 1024**2
    print(f'  {d.name:12s}  {len(imgs):6,} frames  ({mb:.0f} MB)')
    total += len(imgs)
print(f'  TOTAL: {total:,} frames')

if os.path.exists(MANIFEST):
    df = pd.read_csv(MANIFEST)
    print(f'\nManifest rows: {len(df):,}')
    print(df.groupby('location').size().to_string())

_, _, free = shutil.disk_usage(OUTPUT_DIR)
print(f'\nDisk free: {free/1024**3:.2f} GB')